# Stage 0 analysis — free-choice spreading of alternatives

Reads the artifacts written by `python -m src.experiments.run --config <run.yaml>`.
It does **not** run models: every number comes from a cached, provenance-stamped artifact,
and the pooling guard refuses to combine artifacts across devices or dtypes.

**The instrument is pairwise.** Amendment 2 replaced absolute 1–9 ratings with anchor
comparisons fitted by Bradley-Terry, so the item scale is `θ` and the DV is a
designated-versus-other comparison on a **logit** scale. Numbers here are log-odds, never
rating points.

**Primary test** is `(chose − yoked) × |diff|`, predicted negative — the *interaction*. A main
effect of agency is reported but is not the test: alone it is consistent with context-window
sensitivity, which is how the prior version of this claim was eliminated (§1.1).

**Amendment 3 changes what to expect here.** §8's "80% power at the SESOI" is withdrawn as
unsatisfiable (A3.1); the minimum detectable effect is reported instead. `inconclusive` is the
modal outcome across the plausible range, including when the effect is real and exactly at the
SESOI (A3.3) — read it as the design's expected behaviour, not a surprise. The per-model `pass`
rule is unchanged and A3.6 records why. The project gate is revised by A3.7.

The contrast tables below are the table view that accompanies every figure, so series
identity is never conveyed by colour alone.

In [ ]:
import json, sys, warnings
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import arviz as az
import numpy as np
import pandas as pd

from src.analysis import plots, spread_model
from src.config import load_config
from src.experiments import pass_a_pairwise as stage_pw
from src.experiments import pass_b as stage_b, pass_c as stage_c
from src.provenance import assert_poolable, provenance_of, read_parquet
from src.readout import validity

plots.use_paper_defaults()
pd.set_option("display.width", 170, "display.max_columns", 40)

# Point this at the run you want to inspect.
CONFIG = ROOT / "configs" / "stage0_gemma-2-2b.yaml"
cfg = load_config(CONFIG).model_copy(update={"artifacts_dir": ROOT / "artifacts"})
out_dir = cfg.artifact_dir("analysis")

print(cfg.model.name)
print("stage hashes  pass_a", cfg.hash("pass_a"),
      "| pass_b", cfg.hash("pass_b"), "| pass_c", cfg.hash("pass_c"))
print("full config hash", cfg.hash())

## 1. Provenance

All reported numbers must originate on the run machine (CUDA, bf16, pinned revision, clean
tree). `assert_reportable` fails otherwise, and MPS/M1 development artifacts are rejected here
by design — that is what keeps development figures out of the paper.

In [ ]:
comparisons = read_parquet(stage_pw.artifact_path(cfg))   # pairwise Pass A: the instrument
pairs = read_parquet(stage_b.artifact_path(cfg))
trials = read_parquet(stage_c.artifact_path(cfg))

# Hard guard: raises if these came off different devices or dtypes.
assert_poolable([comparisons, pairs, trials], context="notebook load")

prov = provenance_of(trials)
pd.Series(prov).to_frame("value")

In [ ]:
from src.provenance import Provenance, ProvenanceError, assert_reportable

try:
    assert_reportable(Provenance(**prov))
    print("REPORTABLE: run-machine artifact, safe to put in the paper.")
except ProvenanceError as exc:
    print("NOT REPORTABLE (expected for development-machine runs):")
    print(exc)

## 2. The instrument — pairwise Pass A

Three things are checked, and only the third excludes a model.

- **Readout mass** (A1.6) — probability landing on the option-label tokens. Low mass means the
  readout position is not the decision position, i.e. a prompt bug. This is where template
  breakage surfaces first.
- **Order invariance** — reported only. Retired as a gate by A2.2: once the per-template
  position term `β` is in the model, order-reversal consistency carries no information about
  content signal beyond what `θ` already encodes, so testing it against its own model's
  prediction is vacuous (A2.6).
- **Reliability** (A2.2) — empirical split-half of `θ` across disjoint template sets. **The
  sole exclusion criterion.** The model-internal figure is printed beside it for comparison
  only: using it would admit models whose item rankings do not reproduce across paraphrases.
  A2.4 records that the two disagree (0.770 vs 0.957) and that the gap is still open.

In [ ]:
print("readout mass by template (A1.6):")
display(validity.summarize(comparisons, by=["arm", "template"]).round(4))

print("order invariance — reported, not a gate (A2.2/A2.6):")
display(stage_pw.report_order_invariance(comparisons).round(4))

In [ ]:
# The instrument fit is cached by run.py under a key that includes the SOURCE digest of
# bradley_terry.py and pass_a_pairwise.py, because the config hash covers parameters and
# these numbers depend on the estimator's code. Read it rather than refit: refitting here
# would produce a second number with no provenance stamp.
fit_dir = cfg.artifacts_dir / "instrument_fit" / cfg.model.name
records = sorted(fit_dir.glob(f"{cfg.hash('pass_a')}-*/fit_*.json")) if fit_dir.exists() else []
if not records:
    raise FileNotFoundError(
        f"no cached instrument fit under {fit_dir}. Run `python -m src.experiments.run "
        f"--config {CONFIG.name}` first -- this notebook reports, it does not fit.")
instrument = json.loads(records[-1].read_text())
print(instrument["printed"])

In [ ]:
gate = instrument["reliability_gate"]
print(f"empirical split-half of theta ({gate['split_a']} vs {gate['split_b']}): "
      f"Spearman {gate['empirical_reliability_spearman']:.4f}  "
      f"(threshold {gate['threshold']})")
print(f"model-internal reliability: {gate['model_reliability']:.4f}   <- comparison only")
print(f"GATE: {'PASS' if gate['passed'] else 'HALT'}")
for r in gate["reasons"]:
    print("  -", r)

ex = instrument["excess_consistency_slope"]
print(f"\nexcess consistency slope (A2.1 fit check): {ex['slope']:+.4f} "
      f"[{ex['ci_low']:+.4f}, {ex['ci_high']:+.4f}]")
print(f"  vs PPC null {ex['null_mean']:+.4f} (sd {ex['null_sd']:.4f})  ->  "
      f"{ex['z_vs_null']:+.2f} sd")
print("  A non-zero slope means the order model remains misspecified; it exists because of")
print("  R3/R4 and is reported whether or not it is comfortable.")
sigma_item = instrument["sigma_item"]
print(f"\nsigma_item = {sigma_item:.4f}  (the item scale; the SESOI is a fraction of it)")

## 3. Pass B — pairs built from `θ`

Difficulty is *selected* on templates T1–T3 and *analysed* on T4–T5. Selecting on a noisy
`|diff|` and regressing on that same noisy `|diff|` would leave the regressor
regression-contaminated, so the split is disjoint by construction.

Difficult and easy pools are **matched on mean pair rating by design**, not by covariate —
otherwise ceiling compression alone could manufacture the difficulty interaction.

In [ ]:
diag = stage_b.pair_diagnostics(pairs)
display(diag.round(4))
print("matching quality:", {k: round(v, 4) for k, v in diag.attrs.items()})

print(f"\nitem reuse (cap = {cfg.pass_b.max_uses_per_item}), items in both levels (must be 0):")
long = pd.concat([
    pairs[["item1_id", "difficulty"]].rename(columns={"item1_id": "item"}),
    pairs[["item2_id", "difficulty"]].rename(columns={"item2_id": "item"}),
])
print("  max uses:", int(long["item"].value_counts().max()),
      "| items in both levels:", int((long.groupby("item")["difficulty"].nunique() > 1).sum()))

print("\nrealized |diff| by difficulty level, on the ANALYSIS split:")
display(pairs.groupby("difficulty")["diff_analysis"].describe().round(4))

## 4. Power on the realized design — read, not recomputed

`run.py` computes this **between Pass B and Pass C**, so the figure is produced by code that
has not seen a single Pass C outcome. Recomputing it here would produce a second number
without that guarantee, so this cell reads the one the run emitted.

What to look at, per A3.1 and A3.4:

- **`min_detectable_effect` and its ratio to the SESOI.** This replaced "power at the SESOI",
  which A3.1 proved converges to exactly 0.500 at any sample size and can never reach 0.80.
  `power_at_sesoi_not_a_criterion` is printed under that name deliberately.
- **`equivalence_reachable`.** If false, a null for this model may be reported **only** as
  inconclusive, never as equivalence — and that has to be known before the data.
- **`gamma_sensitivity`.** `γ` is the one quantity unknown before Pass C. A3.4 corrected an
  earlier claim that it was not load-bearing: the equivalence branch closes at γ = 1.5.

In [ ]:
res_path = out_dir / f"results_{cfg.hash()}.json"
results = json.loads(res_path.read_text()) if res_path.exists() else {}
power = results.get("power", {})
if not power:
    print(f"no power block in {res_path.name}. It is written between Pass B and Pass C;")
    print("a run stopped before Pass B will not have one.")
elif "skipped" in power:
    print("SKIPPED:", power["skipped"])
else:
    print(f"SE on the primary contrast      {power['se']:.4f}")
    print(f"SESOI (0.15 x sigma_item)       {power['sesoi']:.4f}")
    print(f"minimum detectable effect @80%  {power['min_detectable_effect']:.4f}"
          f"   = {power['mde_over_sesoi']:.3f}x SESOI")
    print(f"equivalence branch reachable    {power['equivalence_reachable']}")
    print(f"power at the SESOI              {power['power_at_sesoi_not_a_criterion']:.3f}"
          "   <- NOT a criterion; see A3.1")
    print(f"section 8                       {power['section_8_criterion']}")
    print("\ninformation by stratum (W = p(1-p) is maximal at p = 0.5, where difficult")
    print("pairs sit by construction -- so easy pairs carry less per observation):")
    display(pd.DataFrame(power["information_by_stratum"]).round(4))
    print("gamma sensitivity:")
    display(pd.DataFrame(power["gamma_sensitivity"]).round(4))

## 5. Pass C — descriptives and position bias

The DV is the **designated-versus-other comparison**, measured pre and post, modelled directly.
There is no per-pair spread anywhere: `spread_model` deliberately exposes no function returning
one, because that quantity's sampling variance is maximal at `p = 0.5` — exactly where difficult
pairs sit by construction — so regressing it on the gap manufactures the predicted interaction
out of noise.

The **pre measurement is shared**: one physical measurement per (pair, template, option order),
emitted once under the `pre` sentinel and reused across all eight conditions. Emitting it per
condition would feed the same observation to the likelihood eight times.

Choice is elicited per (pair, template, option order) and yoking is defined **within** order, so
a flip across orders does not desynchronise `chose` from `yoked`. The flip rate is the natural
measure of how much of the choice is position rather than preference.

In [ ]:
print("readout mass by condition -- a low-mass condition is a prompt bug, not an effect:")
display(validity.summarize(trials, by=["condition"]).round(4))

print("observations by condition x timepoint (pre is emitted ONCE, so it is 1/8 the width):")
display(trials.pivot_table(index="condition", columns="timepoint",
                           values="item1_wins", aggfunc="count", fill_value=0))

In [ ]:
# Position bias: does the model's pick survive reversing the option order?
cell = ["pair_id", "template"]
picks = (trials[trials["condition"] == "chose"]
         .drop_duplicates(cell + ["option_order"])
         .pivot_table(index=cell, columns="option_order",
                      values="chosen_item_id", aggfunc="first"))
if picks.shape[1] == 2:
    flip = (picks.iloc[:, 0] != picks.iloc[:, 1])
    print(f"choice flip rate across option order: {flip.mean():.4f}  (n={len(flip)})")
    print("  0.0 = pure preference, 0.5 = pure position capture")
else:
    print("only one option order present; flip rate undefined")

# Did the discrete choice agree with the model's own pre-manipulation preference?
# Taken from the PRE rows rather than from theta: same instrument, same moment, and it
# costs nothing. Deriving it from theta would mean refitting Bradley-Terry twice here.
pre = (trials[trials["condition"] == spread_model.PRE_SENTINEL]
       .drop_duplicates(cell + ["option_order"])
       .set_index(cell + ["option_order"]))
one = (trials[trials["condition"] == "chose"]
       .drop_duplicates(cell + ["option_order"])
       .set_index(cell + ["option_order"]))
shared = one.index.intersection(pre.index)
agree = ((one.loc[shared, "chosen_item_id"] == one.loc[shared, "item1_id"])
         == pre.loc[shared, "item1_wins"])
print(f"\nchoice agreed with the pre-manipulation preference: {agree.mean():.4f}"
      f"  (n={len(agree)})")
print("  Near 0.5 on difficult pairs is EXPECTED -- that is what makes them difficult,")
print("  and it is also where W = p(1-p) puts the most information.")
display(one.loc[shared].assign(agree=agree).groupby("difficulty")["agree"].mean().round(4))

## 6. The spread model (A2.9.1)

```
logit P(item1 beats item2)
    = u_pair + u_template + beta_t * s + post * d * (gamma_c + lambda_c * diff_z)

  s     +1 if item1 occupies slot 1, -1 otherwise      (position bias)
  d     +1 if item1 is the designated item, -1 otherwise
  post  0 at pre, 1 at post
```

`lambda_c` **is** the interaction — the primary term. `gamma_c` is the post shift at mean
`|diff|`, reported but never primary.

Two corrections to A2.9.1's written form, both identification issues found on implementation
and both stated rather than silently applied: `delta_pair` and `u_pair` are the same quantity
and are fitted as one; and the outcome is oriented on a **fixed pair axis** (item1 vs item2,
canonical by sorted id) with designation carried in the sign `d`, so the per-pair baseline
cannot absorb part of the selection artifact.

In [ ]:
design = spread_model.prepare(cfg, trials)
print(f"|diff| centring: mean={design.diff_mean:.4f} sd={design.diff_sd:.4f}")
print(f"conditions fitted ({len(design.conditions)}):", design.conditions)

posterior_path = out_dir / f"spread_posterior_{cfg.hash()}.nc"
if posterior_path.exists():
    idata = az.from_netcdf(str(posterior_path))
    print("loaded cached posterior:", posterior_path.name)
else:
    print("no cached posterior; fitting (this is the expensive cell)")
    idata = spread_model.fit(cfg, design, progressbar=True)

conv = spread_model.convergence(cfg, idata)
bad = conv[~(conv["rhat_ok"] & conv["ess_ok"])]
print(f"convergence failures: {len(bad)}  (reported, never silently re-tuned -- section 7.1)")
display(bad if len(bad) else conv.head(12).round(4))

## 7. Planned contrasts

| contrast | what it isolates |
|---|---|
| `chose − yoked` | **PRIMARY** — the full manipulation, designation held constant |
| `chose − self-recounted` | transcript structure at the self-attributed wording |
| `structure-control − yoked` | transcript structure at the unattributed wording |
| `self-recounted − yoked` | wording with structure held constant |
| `chose − chose-provisional` | reversibility — dissonance predicts an effect, self-perception does not (A2.8) |
| `chose − 3p-yoked` | self-attribution, information held constant |
| `3p-yoked − yoked` | information effect, designation held constant |
| `yoked − random` | selection artifact |
| `3p-random − random` | pure context effect — the rebuttal's mechanism |

The rebuttal's account predicts the effect lives in `3p-yoked − yoked` and `3p-random − random`
and that **the primary interaction is null**. Consistency restoration predicts the interaction
is present regardless of those two.

A2.9.3's correction: the term *authorship* is retired. A transformer retains no record of having
produced a token, so model-generated text fed back as context is processed identically to
experimenter-supplied text. These contrasts isolate **role-attribution in the transcript**, not
production.

In [ ]:
sesoi = cfg.analysis.sesoi_sigma_fraction * sigma_item
print(f"SESOI = {cfg.analysis.sesoi_sigma_fraction} x sigma_item ({sigma_item:.4f}) "
      f"= {sesoi:.4f} logits per SD of |diff|")
print("A2.9.2 retired the fixed raw-point anchor: it has no meaning on a logit scale.\n")

contrasts = spread_model.contrasts(cfg, idata, sesoi)
print("INTERACTION -- lambda, the primary term:")
display(contrasts[contrasts.term == "lambda"].round(4).set_index("name"))
print("POST SHIFT at mean |diff| -- gamma, reported but never primary:")
display(contrasts[contrasts.term == "gamma"].round(4).set_index("name"))

In [ ]:
primary = contrasts[(contrasts.name == spread_model.PRIMARY)
                    & (contrasts.term == "lambda")].iloc[0]
print(f"PRIMARY TEST  {spread_model.PRIMARY} x |diff|  ->  {primary['decision'].upper()}")
print(f"  median {primary['median']:+.4f}   "
      f"{int(cfg.analysis.hdi_prob * 100)}% HDI "
      f"[{primary['hdi_low']:+.4f}, {primary['hdi_high']:+.4f}]")
print(f"  P(<0) = {primary['p_negative']:.4f}")
print(f"  exceeds SESOI: {bool(primary['exceeds_sesoi'])}   "
      f"inside ROPE: {bool(primary['inside_rope'])}")
print("\n9.2 cells, unamended (A3.6 declined to loosen `pass`):")
print("  pass          HDI excludes 0 in the predicted (negative) direction")
print("                AND |median| > SESOI")
print("  fail          the whole HDI lies inside [-SESOI, +SESOI]  (equivalence)")
print("  inconclusive  neither -- the MODAL outcome across the plausible range (A3.3)")

agree = spread_model.structure_factor_agreement(contrasts)
if agree:
    print("\nTranscript structure estimated twice, at both wording levels (A2.9.3):")
    print(f"  {agree['edge_a']} = {agree['estimate_a']:+.4f}")
    print(f"  {agree['edge_b']} = {agree['estimate_b']:+.4f}")
    print(f"  discrepancy = {agree['discrepancy']:+.4f}")
    print("  Agreement means a turn-presence effect is identified and subtractable.")
    print("  Disagreement means structure interacts with wording and NEITHER edge is")
    print("  interpretable alone -- which is a result about the design, not about H1.")

## 8. Figures

Hue carries **attribution** (three levels, validated for colourblind separation at all pairs in
both light and dark) and line style carries transcript structure and designation. Eight
conditions cannot be eight hues; that composite encoding also happens to match A2.9.3's
factorial structure. Every series is direct-labelled, so identity never depends on colour, and
the contrast tables above are the table view.

The points on the interaction panels are **bin-pooled** empirical shifts, descriptive only.
They are not per-pair spreads — see §5 and `plots._empirical_shift` for why that distinction
decides whether the data layer is informative or self-fulfilling.

In [ ]:
(out_dir / "figures").mkdir(parents=True, exist_ok=True)

fig = plots.interaction_plot(cfg, design, idata)
plots.save(fig, out_dir / "figures" / f"nb_interaction_{cfg.hash()}.png")
fig

In [ ]:
fig = plots.forest_plot(cfg, contrasts, sesoi, term="lambda")
plots.save(fig, out_dir / "figures" / f"nb_forest_lambda_{cfg.hash()}.png")
fig

In [ ]:
fig = plots.forest_plot(cfg, contrasts, sesoi, term="gamma")
plots.save(fig, out_dir / "figures" / f"nb_forest_gamma_{cfg.hash()}.png")
fig

## 9. §7.2's item random effect — withdrawn as specified (A3.8)

§7.2 preregisters a robustness model adding `u_item[item1] + u_item[item2]`. It is **not fitted
here**, and the reason is recorded rather than the section quietly dropped.

The sum form was written for a per-pair spread in rating points, where both items contribute
additively to one number. On the current fixed pair axis item quality enters as a
**difference**, `u_item[item1] − u_item[item2]`, and that difference is separable from the free
per-pair `u_pair` only through items appearing in more than one pair. Item reuse is capped at
**two**, so each `u_item` would be informed by at most two pairs whose own `u_pair` is free —
the posterior would be dominated by the prior, and §7.2's "the discrepancy is reported either
way" would report prior sensitivity rather than robustness.

The concern has not been abandoned. The item scale is `θ`, estimated on the Bradley-Terry
instrument from Pass A's anchor comparisons where every item appears many times and
identification is not in question, and `|diff|` is a function of exactly that quantity. Any
replacement must be specified against the logit DV with its identification demonstrated first.
See **A3.8**.

## 10. The ladder and the project gate

Whether the interaction emerges with scale is a primary descriptive result **regardless of the
gate outcome**.

**Project gate, as revised by A3.7:** Stage 1 is entered if at least one non-excluded model
passes §9.2 in full, **and** at least two are directional at `P(λ < 0 | data) ≥ 0.95`. The old
"at least two models pass" rule inherited A3.1's cap — it opened Stage 1 only 49.3% of the time
on a real effect exactly at the SESOI. The per-model `pass` rule was left untouched (A3.6).

Models excluded by the reliability gate are reported as *not having demonstrated that their item
rankings reproduce across paraphrases*. That is not evidence about the hypotheses in either
direction.

In [ ]:
LADDER = ["stage0_qwen2.5-0.5b.yaml", "stage0_qwen2.5-1.5b.yaml", "stage0_qwen2.5-3b.yaml",
          "stage0_gemma-2-2b.yaml", "stage0_llama-3.2-3b.yaml"]

rows = []
for name in LADDER:
    c = load_config(ROOT / "configs" / name).model_copy(
        update={"artifacts_dir": ROOT / "artifacts"})
    res = c.artifact_dir("analysis") / f"results_{c.hash()}.json"
    if not res.exists():
        rows.append({"model": c.model.name, "status": "not run"})
        continue
    r = json.loads(res.read_text())
    g, p, pw = r.get("reliability_gate", {}), r.get("primary", {}), r.get("power", {})
    rows.append({
        "model": c.model.name,
        "status": r.get("outcome", "?"),
        "device": (r.get("provenance") or {}).get("device"),
        "reliability": g.get("empirical_reliability_spearman"),
        "gate": g.get("passed"),
        "sesoi": r.get("sesoi"),
        "mde/sesoi": pw.get("mde_over_sesoi"),
        "equiv_reachable": pw.get("equivalence_reachable"),
        "lambda_median": p.get("median"),
        "hdi_low": p.get("hdi_low"),
        "hdi_high": p.get("hdi_high"),
        "p_negative": p.get("p_negative"),
        "decision": p.get("decision"),
    })
ladder = pd.DataFrame(rows)
display(ladder.round(4))

In [ ]:
# A3.7's gate, evaluated explicitly rather than eyeballed from the table.
TAU = 0.95
run_rows = ladder[ladder["decision"].notna()]
full_pass = run_rows[run_rows["decision"] == "pass"]
directional = run_rows[run_rows["p_negative"].fillna(0) >= TAU]

print(f"models with a full 9.2 pass: {len(full_pass)}  {list(full_pass['model'])}")
print(f"models directional at P(lambda<0) >= {TAU}: {len(directional)}  "
      f"{list(directional['model'])}")
entered = len(full_pass) >= 1 and len(directional) >= 2
print(f"\nA3.7 project gate -> Stage 1 {'ENTERED' if entered else 'NOT entered'}")
if not entered:
    print("  Reported as a negative result. The ladder above is still a primary")
    print("  descriptive result, and the Stage 1 code named in section 11 is not written.")
excluded = run_rows[run_rows["gate"] == False]  # noqa: E712
if len(excluded):
    print(f"\nexcluded by the reliability gate: {list(excluded['model'])}")
    print("  -> did not demonstrate that item rankings reproduce across paraphrases;")
    print("     not evidence about H1 in either direction.")